# Label Generator — Triple Barrier Method
Applies ternary `Target` labels to every ticker in `data/merged_raw/` and writes
the result to `data/labeled/`.

| Barrier | Condition | Label |
|---|---|---|
| Take-Profit (TP) | Forward return ≥ `tp_pct` hits first | `1` |
| Stop-Loss (SL) | Forward return ≤ `sl_pct` hits first | `-1` |
| Time expiry | Neither barrier hit within `horizon` days | `0` |

The last `horizon` rows per ticker are dropped — they have no fully observable future window.

> **Implementation note:** The barrier check is fully vectorised via NumPy
> `sliding_window_view`. No `iterrows`, `apply`, or Python DataFrame loops are used.

## Cell 1 — Imports & Path Configuration

In [11]:
import os
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ---------------------------------------------------------------------------
# Path constants
# ---------------------------------------------------------------------------
MERGED_DIR: str = "data/merged_raw"
LABELED_DIR: str = "data/labeled"

# ---------------------------------------------------------------------------
# Triple Barrier hyperparameters
# ---------------------------------------------------------------------------
HORIZON: int = 5       # trading days to look forward
TP_PCT: float = 0.04   # +3 % take-profit barrier
SL_PCT: float = -0.03  # -3 % stop-loss barrier

Path(LABELED_DIR).mkdir(parents=True, exist_ok=True)

print(f"Input  : {MERGED_DIR}")
print(f"Output : {LABELED_DIR}")
print(f"Params : horizon={HORIZON}  TP={TP_PCT:+.1%}  SL={SL_PCT:+.1%}")

Input  : data/merged_raw
Output : data/labeled
Params : horizon=5  TP=+4.0%  SL=-3.0%


## Cell 2 — Triple Barrier Labeling Function

**Vectorised implementation** — no `iterrows`, no Python DataFrame loops.

For each row `i`, the core steps are:
1. Build a `(N - horizon) × horizon` matrix of forward `adjusted_close` prices using `sliding_window_view`.
2. Compute percentage returns relative to each row's entry price in one broadcast operation.
3. Find the **first** TP hit and **first** SL hit per row with `np.argmax` on the boolean masks.
4. Label `1` where the TP index strictly precedes the SL index (and TP was actually reached).

In [12]:
def apply_triple_barrier(
    df: pd.DataFrame,
    horizon: int = HORIZON,
    tp_pct: float = TP_PCT,
    sl_pct: float = SL_PCT,
) -> pd.DataFrame:
    """
    Apply the Triple Barrier Method using intraday High/Low prices.

    For each bar i, the entry price is `adjusted_close[i]`. The forward window
    [i+1, i+horizon] is scanned using each day's `high` (for TP) and `low`
    (for SL) to determine the first barrier touched.

    First-touch tie-breaking: if both barriers are touched on the same day,
    SL wins (conservative).

    Parameters
    ----------
    df : pd.DataFrame
        Merged ticker DataFrame with columns: Date, adjusted_close, high, low.
    horizon : int
        Number of trading days in the forward-looking window.
    tp_pct : float
        Take-profit barrier as a fractional return (e.g. 0.03 = +3 %).
    sl_pct : float
        Stop-loss barrier as a fractional return (e.g. -0.03 = -3 %).
        Must be negative.

    Returns
    -------
    pd.DataFrame
        Original DataFrame (minus the last `horizon` rows) with a new
        integer `Target` column:
          1  = TP (high) hit first
         -1  = SL (low) hit first  (ties on same day → SL)
          0  = time expiry (neither barrier reached within horizon)

    Raises
    ------
    KeyError
        If any of adjusted_close, high, or low are absent from `df`.
    ValueError
        If the DataFrame has fewer rows than `horizon + 1`.
    """
    required: list[str] = ["adjusted_close", "high", "low"]
    missing_cols: list[str] = [c for c in required if c not in df.columns]
    if missing_cols:
        raise KeyError(
            f"Missing required columns: {missing_cols}. "
            f"Available: {df.columns.tolist()}"
        )

    df = df.sort_values("Date").copy() if "Date" in df.columns else df.sort_index().copy()

    n: int = len(df)
    if n <= horizon:
        raise ValueError(
            f"DataFrame has only {n} rows — need at least horizon+1={horizon + 1}."
        )

    entry_prices: np.ndarray = df["adjusted_close"].to_numpy(dtype=np.float64)
    highs: np.ndarray = df["high"].to_numpy(dtype=np.float64)
    lows: np.ndarray  = df["low"].to_numpy(dtype=np.float64)

    n_labeled: int = n - horizon

    # Build forward High/Low windows: shape (n_labeled, horizon)
    # Row i → [highs[i+1], ..., highs[i+horizon]]
    future_highs: np.ndarray = sliding_window_view(highs[1:], horizon)[:n_labeled]
    future_lows: np.ndarray  = sliding_window_view(lows[1:],  horizon)[:n_labeled]

    # Entry price broadcast: (n_labeled, 1)
    ep: np.ndarray = entry_prices[:n_labeled, np.newaxis]

    # Fractional returns using intraday extremes
    high_returns: np.ndarray = (future_highs - ep) / ep   # (n_labeled, horizon)
    low_returns: np.ndarray  = (future_lows  - ep) / ep   # (n_labeled, horizon)

    NEVER: int = horizon  # sentinel — "barrier never touched"

    tp_mask: np.ndarray = high_returns >= tp_pct   # high breaches TP
    sl_mask: np.ndarray = low_returns  <= sl_pct   # low  breaches SL

    tp_first: np.ndarray = np.where(tp_mask.any(axis=1), np.argmax(tp_mask, axis=1), NEVER)
    sl_first: np.ndarray = np.where(sl_mask.any(axis=1), np.argmax(sl_mask, axis=1), NEVER)

    # TP wins only if it fires STRICTLY before SL.
    # Same-day hit → SL wins (conservative tie-breaking).
    labels: np.ndarray = np.select(
        condlist=[
            (tp_first < sl_first) & (tp_first < NEVER),   # TP first
            (sl_first <= tp_first) & (sl_first < NEVER),  # SL first (or tie)
        ],
        choicelist=[1, -1],
        default=0,
    ).astype(np.int8)

    labeled_df: pd.DataFrame = df.iloc[:n_labeled].copy()
    labeled_df["Target"] = labels
    return labeled_df

## Cell 3 — Process & Export All Tickers

In [13]:
input_files: list[str] = sorted(glob(os.path.join(MERGED_DIR, "*_merged.csv")))

if not input_files:
    print(f"[WARN] No merged CSVs found in {MERGED_DIR}. Run data_merger.ipynb first.")

skipped: list[str] = []
total: int = len(input_files)

print(f"Labeling {total} tickers  (horizon={HORIZON}, TP={TP_PCT:+.1%}, SL={SL_PCT:+.1%})\n")
print(f"{'Ticker':<8}  {'Rows':>6}  {'TP=1':>6}  {'SL=-1':>7}  {'Exp=0':>7}  {'TP Rate':>8}")
print("-" * 54)

for filepath in input_files:
    # Extract ticker symbol from filename: "AAPL_merged.csv" → "AAPL"
    ticker: str = os.path.basename(filepath).replace("_merged.csv", "")

    try:
        df: pd.DataFrame = pd.read_csv(filepath, parse_dates=["Date"])
    except (OSError, ValueError) as exc:
        print(f"[ERROR] {ticker}: could not load CSV — {exc}")
        skipped.append(ticker)
        continue

    try:
        labeled: pd.DataFrame = apply_triple_barrier(df, HORIZON, TP_PCT, SL_PCT)
    except (KeyError, ValueError) as exc:
        print(f"[ERROR] {ticker}: labeling failed — {exc}")
        skipped.append(ticker)
        continue

    out_path: str = os.path.join(LABELED_DIR, f"{ticker}_labeled.csv")
    labeled.to_csv(out_path, index=False)

    # ------------------------------------------------------------------
    # Label distribution logging
    # ------------------------------------------------------------------
    n_rows: int = len(labeled)
    n_tp: int = int((labeled["Target"] == 1).sum())
    n_sl: int = int((labeled["Target"] == -1).sum())
    n_exp: int = int((labeled["Target"] == 0).sum())
    tp_rate: float = n_tp / n_rows if n_rows > 0 else 0.0

    print(f"{ticker:<8}  {n_rows:>6}  {n_tp:>6}  {n_sl:>7}  {n_exp:>7}  {tp_rate:>7.1%}")

# Summary
print("-" * 54)
saved: int = total - len(skipped)
print(f"\nDone. {saved}/{total} tickers labeled and saved to '{LABELED_DIR}/'.")
if skipped:
    print(f"Skipped: {skipped}")

Labeling 50 tickers  (horizon=5, TP=+4.0%, SL=-3.0%)

Ticker      Rows    TP=1    SL=-1    Exp=0   TP Rate
------------------------------------------------------
AAPL        2505    1750      284      471    69.9%
ADBE        2508     701      925      882    28.0%
AMAT        2501    2103      285      113    84.1%
AMC         2464     240     2207       17     9.7%
AMD         2508    1126     1283       99    44.9%
AMZN        2506    1830      404      272    73.0%
ARM          658     305      346        7    46.4%
ASML        2481    2111      230      140    85.1%
AVGO        2491    2272      194       25    91.2%
COIN        1268     498      768        2    39.3%
CRM         2501    1204      614      683    48.1%
CRWD        1731     769      887       75    44.4%
DDOG        1662     697      863      102    41.9%
GME         2202    1626      524       52    73.8%
GOOGL       2503    1946      289      268    77.7%
HOOD        1191     489      682       20    41.1%
HUBS  